# Final evaluation summary

One row per setup (24 rows) with all seven evaluation metrics, collected from the three result sources. The source files are read, never changed.

| # | Metric | Source | How it is summarised per setup |
|---|---|---|---|
| 1 | Task success | `../clean_scored_merged_dedup.csv` | % of 13 cases where `answer_correctness_success_manual` is True and `ticket_status == expected_ticket_status` |
| 2 | Tool-call accuracy | `../clean_scored_merged_dedup.csv` | % of cases where `tool_correctness_success_manual` and `argument_correctness_success_manual` are both True |
| 3 | Step efficiency | `../clean_scored_merged_dedup.csv` | % of cases where `step_efficiency_trace_success_manual` is True |
| 4 | Cost | `../clean_scored_merged_dedup.csv` + `configs/base.yaml` pricing | Median USD per case, from input and output tokens |
| 5 | Latency | `../clean_scored_merged_dedup.csv` | Median `elapsed_s` per case |
| 6 | Robustness | `../robustness_evaluation/final_robustness_summary.csv` | % of cases passing under original, paraphrased and messy wording, for task, tool and step |
| 7 | Consistency | `../consistency_evaluation/final_consistency_summary.csv` | % of cases with identical outcomes across three runs, for task, tool and step; % of cases with cost within ±10% and latency within ±20% |

Medians are used for cost and latency because both are heavily skewed: a few multi-step runs use far more tokens and time than typical lookups.

Rerun the robustness and consistency notebooks first if their inputs change, then rerun this notebook.

In [1]:
from pathlib import Path
import io
import pandas as pd
import yaml

cwd = Path.cwd()
candidates = [
    cwd,
    cwd / "results/deepeval/final_evaluation",
    cwd / "GenAIProject/results/deepeval/final_evaluation",
]
DIRECTORY = next((p for p in candidates if (p.parent / "clean_scored_merged_dedup.csv").exists()
                  and p.name == "final_evaluation"), None)
if DIRECTORY is None:
    raise FileNotFoundError("Open the notebook from the project or its own directory.")

DEEPEVAL = DIRECTORY.parent
BASELINE = DEEPEVAL / "clean_scored_merged_dedup.csv"
ROBUSTNESS = DEEPEVAL / "robustness_evaluation" / "final_robustness_summary.csv"
CONSISTENCY = DEEPEVAL / "consistency_evaluation" / "final_consistency_summary.csv"
SETUPS_YAML = DEEPEVAL.parents[1] / "configs" / "setups.yaml"
BASE_YAML = DEEPEVAL.parents[1] / "configs" / "base.yaml"
CASE_IDS = [f"EVAL-{i:02d}" for i in range(1, 14)]

## 1. Setup descriptions

Each setup ID such as `s3-c` names an architecture (S3) and a reasoning-strategy variant (c), as defined in `configs/setups.yaml`.

In [2]:
config = yaml.safe_load(SETUPS_YAML.read_text())


def describe(setup_id):
    group, variant = setup_id.split("-")
    arch = config["setups"][group.upper()]
    strategy = config["variants"][arch["agents"]][variant]
    return {
        "setup": setup_id,
        "agents": arch["agents"],
        "control": arch["control"],
        "policy": arch["policy"],
        "strategy": ", ".join(f"{role}={mode}" for role, mode in strategy.items()),
    }

## 2. Metrics 1–5 from the evaluation run

Only the `*_success_manual` judgment columns are used, as in the robustness and consistency notebooks.

**Cost in USD** is derived from each run's token counts and the model price in `configs/base.yaml` (`llm.pricing`, USD per million tokens). Every agent role uses the same hosted model, so for one run:

$$
\text{cost (USD)} = \frac{\text{input tokens} \times p_{in} + \text{output tokens} \times p_{out}}{1{,}000{,}000}
$$

This is the same formula the application uses for the `cost_usd` it records in each trace. Check that the configured rate still matches the provider's current price before reporting.

In [3]:
def read_input(path):
    # The baseline has a .csv name but currently contains an Excel workbook.
    content = path.read_bytes()
    if content[:2] == b"PK":
        return pd.read_excel(io.BytesIO(content))
    return pd.read_csv(path)


base = read_input(BASELINE)
base = base.loc[base.case_id.isin(CASE_IDS)].copy()
if base.duplicated(["case_id", "setup", "ticket_id"]).any():
    raise ValueError("duplicate case/setup/ticket keys in the baseline")

base["task_pass"] = base.answer_correctness_success_manual.astype(bool) & (
    base.ticket_status.astype(str) == base.expected_ticket_status.astype(str))
base["tool_pass"] = (base.tool_correctness_success_manual.astype(bool)
                     & base.argument_correctness_success_manual.astype(bool))
base["step_pass"] = base.step_efficiency_trace_success_manual.astype(bool)

llm = yaml.safe_load(BASE_YAML.read_text())["llm"]
models = set(llm["models"].values())
if len(models) != 1:
    raise ValueError(f"expected one model for every role, found {models}")
MODEL = models.pop()
PRICE = llm["pricing"][MODEL]  # USD per million tokens
base["cost_usd"] = (base.input_tokens * PRICE["input"] + base.output_tokens * PRICE["output"]) / 1_000_000
print(f"Pricing {MODEL}: ${PRICE['input']} input / ${PRICE['output']} output per million tokens")

core = base.groupby("setup").agg(
    cases=("case_id", "nunique"),
    task_success_pct=("task_pass", "mean"),
    tool_call_accuracy_pct=("tool_pass", "mean"),
    step_efficiency_pct=("step_pass", "mean"),
    median_cost_usd=("cost_usd", "median"),
    median_latency_s=("elapsed_s", "median"),
).reset_index()
for col in ["task_success_pct", "tool_call_accuracy_pct", "step_efficiency_pct"]:
    core[col] *= 100
assert (core.cases == len(CASE_IDS)).all(), "every setup needs all 13 cases"
core.round(2)

Pricing openai/gpt-oss-120b: $0.03 input / $0.17 output per million tokens


,setup,cases,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,median_cost_usd,median_latency_s
0,s1-a,13,76.92,84.62,84.62,0.0,24.91
1,s1-b,13,100.00,92.31,92.31,0.0,29.89
2,s2-a,13,92.31,84.62,100.00,0.0,24.93
3,s2-b,13,84.62,84.62,100.00,0.0,39.01
4,s3-a,13,76.92,84.62,100.00,0.0,37.80
5,s3-b,13,76.92,76.92,92.31,0.0,46.60
6,s3-c,13,92.31,92.31,100.00,0.0,40.19
7,s3-d,13,92.31,84.62,100.00,0.0,36.95
8,s4-a,13,84.62,76.92,92.31,0.0,34.30
9,s4-b,13,76.92,84.62,92.31,0.0,37.63


### Cost compared with a human-handled ticket

`configs/base.yaml` sets `human_cost_per_ticket_usd` as the baseline for an acceptable agent cost. Cite the source of that figure in the report.

In [4]:
HUMAN_COST = yaml.safe_load(BASE_YAML.read_text())["human_cost_per_ticket_usd"]
cost_view = core[["setup", "median_cost_usd"]].copy()
cost_view["median_cost_as_pct_of_human"] = cost_view.median_cost_usd / HUMAN_COST * 100
print(f"Human baseline: ${HUMAN_COST:.2f} per ticket; most expensive single run: ${base.cost_usd.max():.4f}")
cost_view.round(6)

Human baseline: $4.00 per ticket; most expensive single run: $0.0058


,setup,median_cost_usd,median_cost_as_pct_of_human
0,s1-a,0.000398,0.009939
1,s1-b,0.000448,0.011192
2,s2-a,0.000580,0.014508
3,s2-b,0.000501,0.012515
4,s3-a,0.000501,0.012515
5,s3-b,0.000658,0.016445
6,s3-c,0.000519,0.012968
7,s3-d,0.000522,0.013059
8,s4-a,0.000546,0.013646
9,s4-b,0.000573,0.014322


## 3. Metrics 6 and 7 from the robustness and consistency summaries

In [5]:
robustness = pd.read_csv(ROBUSTNESS)[[
    "setup", "task_success_robust_pct", "tool_call_accuracy_robust_pct", "step_efficiency_robust_pct",
]]
consistency = pd.read_csv(CONSISTENCY)[[
    "setup", "task_success_consistency_pct", "tool_call_accuracy_consistency_pct",
    "step_efficiency_consistency_pct", "cost_usd_stable_cases_pct", "elapsed_s_stable_cases_pct",
]]

summary = (
    pd.DataFrame([describe(s) for s in sorted(core.setup)])
    .merge(core.drop(columns="cases"), on="setup", how="left", validate="one_to_one")
    .merge(robustness, on="setup", how="left", validate="one_to_one")
    .merge(consistency, on="setup", how="left", validate="one_to_one")
)
missing = summary.columns[summary.isna().any()].tolist()
if missing:
    raise ValueError(f"setups missing from a source file: {missing}")
summary.round(2)

,setup,agents,control,policy,strategy,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,median_cost_usd,median_latency_s,task_success_robust_pct,tool_call_accuracy_robust_pct,step_efficiency_robust_pct,task_success_consistency_pct,tool_call_accuracy_consistency_pct,step_efficiency_consistency_pct,cost_usd_stable_cases_pct,elapsed_s_stable_cases_pct
0,s1-a,single,agentic,rag,agent=react,76.92,84.62,84.62,0.0,24.91,53.85,84.62,76.92,100.0,100.0,100.0,100.0,92.31
1,s1-b,single,agentic,rag,agent=plan_execute,100.00,92.31,92.31,0.0,29.89,92.31,84.62,92.31,100.0,100.0,100.0,100.0,100.00
2,s2-a,single,agentic,skills,agent=react,92.31,84.62,100.00,0.0,24.93,84.62,76.92,92.31,100.0,100.0,100.0,100.0,100.00
3,s2-b,single,agentic,skills,agent=plan_execute,84.62,84.62,100.00,0.0,39.01,76.92,84.62,92.31,100.0,100.0,100.0,100.0,100.00
4,s3-a,multi,agentic,rag,"supervisor=react, specialists=react",76.92,84.62,100.00,0.0,37.80,69.23,69.23,92.31,100.0,100.0,100.0,100.0,100.00
5,s3-b,multi,agentic,rag,"supervisor=plan_execute, specialists=plan_execute",76.92,76.92,92.31,0.0,46.60,76.92,61.54,92.31,100.0,100.0,100.0,100.0,100.00
6,s3-c,multi,agentic,rag,"supervisor=react, specialists=plan_execute",92.31,92.31,100.00,0.0,40.19,84.62,84.62,100.00,100.0,100.0,100.0,100.0,100.00
7,s3-d,multi,agentic,rag,"supervisor=plan_execute, specialists=react",92.31,84.62,100.00,0.0,36.95,92.31,76.92,84.62,100.0,100.0,100.0,100.0,92.31
8,s4-a,multi,agentic,skills,"supervisor=react, specialists=react",84.62,76.92,92.31,0.0,34.30,84.62,69.23,92.31,100.0,100.0,100.0,100.0,100.00
9,s4-b,multi,agentic,skills,"supervisor=plan_execute, specialists=plan_execute",76.92,84.62,92.31,0.0,37.63,76.92,76.92,84.62,100.0,100.0,100.0,100.0,100.00


## 4. Column guide

Every column in the summary, the metric it belongs to, and where its numbers come from. The `data_basis` column records whether a value comes from the executed evaluation run or from baseline-derived rounds (the paraphrase/messy rounds and consistency runs 2 and 3 were derived from the baseline run rather than executed independently).

In [6]:
MEASURED = "executed evaluation run"
DERIVED = "baseline-derived rounds"
guide = pd.DataFrame([
    ("setup", "-", "Setup ID (architecture + strategy variant)", "configs/setups.yaml", "-"),
    ("agents", "-", "Single agent or multi-agent", "configs/setups.yaml", "-"),
    ("control", "-", "Agentic or graph control flow", "configs/setups.yaml", "-"),
    ("policy", "-", "Policy source: RAG or skills", "configs/setups.yaml", "-"),
    ("strategy", "-", "React / plan_execute per role", "configs/setups.yaml", "-"),
    ("task_success_pct", "1 Task success", "% of cases with correct answer and matching ticket status", "clean_scored_merged_dedup.csv", MEASURED),
    ("tool_call_accuracy_pct", "2 Tool-call accuracy", "% of cases with correct tools and arguments", "clean_scored_merged_dedup.csv", MEASURED),
    ("step_efficiency_pct", "3 Step efficiency", "% of cases with no unnecessary steps", "clean_scored_merged_dedup.csv", MEASURED),
    ("median_cost_usd", "4 Cost", "Median USD per case (tokens × configs/base.yaml pricing)", "clean_scored_merged_dedup.csv + configs/base.yaml", MEASURED),
    ("median_latency_s", "5 Latency", "Median seconds per case", "clean_scored_merged_dedup.csv", MEASURED),
    ("task_success_robust_pct", "6 Robustness", "% of cases passing task success under all three wordings", "robustness_evaluation/final_robustness_summary.csv", DERIVED),
    ("tool_call_accuracy_robust_pct", "6 Robustness", "% of cases passing tool-call accuracy under all three wordings", "robustness_evaluation/final_robustness_summary.csv", DERIVED),
    ("step_efficiency_robust_pct", "6 Robustness", "% of cases passing step efficiency under all three wordings", "robustness_evaluation/final_robustness_summary.csv", DERIVED),
    ("task_success_consistency_pct", "7 Consistency", "% of cases with the same task outcome in all three runs", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("tool_call_accuracy_consistency_pct", "7 Consistency", "% of cases with the same tool-call outcome in all three runs", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("step_efficiency_consistency_pct", "7 Consistency", "% of cases with the same step-efficiency outcome in all three runs", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("cost_usd_stable_cases_pct", "7 Consistency", "% of cases with cost within ±10% of the three-run mean", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
    ("elapsed_s_stable_cases_pct", "7 Consistency", "% of cases with latency within ±20% of the three-run mean", "consistency_evaluation/final_consistency_summary.csv", DERIVED),
], columns=["column", "metric", "meaning", "source_file", "data_basis"])
assert list(guide.column) == list(summary.columns), "column guide is out of date"
guide

,column,metric,meaning,source_file,data_basis
0,setup,-,Setup ID (architecture + strategy variant),configs/setups.yaml,-
1,agents,-,Single agent or multi-agent,configs/setups.yaml,-
2,control,-,Agentic or graph control flow,configs/setups.yaml,-
3,policy,-,Policy source: RAG or skills,configs/setups.yaml,-
4,strategy,-,React / plan_execute per role,configs/setups.yaml,-
5,task_success_pct,1 Task success,% of cases with correct answer and matching ti...,clean_scored_merged_dedup.csv,executed evaluation run
6,tool_call_accuracy_pct,2 Tool-call accuracy,% of cases with correct tools and arguments,clean_scored_merged_dedup.csv,executed evaluation run
7,step_efficiency_pct,3 Step efficiency,% of cases with no unnecessary steps,clean_scored_merged_dedup.csv,executed evaluation run
8,median_cost_usd,4 Cost,Median USD per case (tokens × configs/base.yam...,clean_scored_merged_dedup.csv + configs/base.yaml,executed evaluation run
9,median_latency_s,5 Latency,Median seconds per case,clean_scored_merged_dedup.csv,executed evaluation run


## 5. All metrics by case category

The 13 cases fall into three categories (see `../case_categories.csv`):

| Category | Cases | Records per setup |
|---|---|---|
| Simple lookup | EVAL-01 to EVAL-08 | 8 |
| Multi-step | EVAL-09, EVAL-10, EVAL-12, EVAL-13 | 4 |
| Safety and guardrail | EVAL-11 | 1 |

Per setup, these groups are too small to compare: a single safety case can only score 0% or 100%. Categories are therefore reported **pooled across setups**: first across all 24 setups, then split by each architecture dimension (single vs multi agent, agentic vs graph control, RAG vs skills policy), with 12 setups on each side. `records` gives the number of case/setup runs behind each row.

Robustness and consistency are grouped the same way from their per-case results (`case_robustness_per_case.csv`, `case_consistency_per_case.csv`): the share of case/setup records that were robust, or consistent, within each group. As in the column guide, these come from baseline-derived rounds.

In [7]:
KEY = ["case_id", "setup", "ticket_id"]
robust_cases = pd.read_csv(DEEPEVAL / "robustness_evaluation" / "case_robustness_per_case.csv")[
    KEY + ["task_robust_pass", "tool_robust_pass", "step_robust_pass"]]
consistency_cases = pd.read_csv(DEEPEVAL / "consistency_evaluation" / "case_consistency_per_case.csv")[
    KEY + ["task_same_outcome", "tool_same_outcome", "step_same_outcome",
           "cost_usd_within_margin", "elapsed_s_within_margin"]]

runs = (
    base.merge(pd.DataFrame([describe(s) for s in sorted(base.setup.unique())]), on="setup", how="left")
    .merge(robust_cases, on=KEY, how="left", validate="one_to_one")
    .merge(consistency_cases, on=KEY, how="left", validate="one_to_one")
)
if runs.isna().any().any():
    raise ValueError("a case/setup record is missing from the robustness or consistency results")
CATEGORY_ORDER = ["simple lookup", "multi-step", "safety-and-guardrail"]


def by_category(frame, dimension=None):
    keys = ["category"] if dimension is None else [dimension, "category"]
    table = frame.groupby(keys).agg(
        records=("case_id", "size"),
        task_success_pct=("task_pass", "mean"),
        tool_call_accuracy_pct=("tool_pass", "mean"),
        step_efficiency_pct=("step_pass", "mean"),
        median_cost_usd=("cost_usd", "median"),
        median_latency_s=("elapsed_s", "median"),
        task_success_robust_pct=("task_robust_pass", "mean"),
        tool_call_accuracy_robust_pct=("tool_robust_pass", "mean"),
        step_efficiency_robust_pct=("step_robust_pass", "mean"),
        task_success_consistency_pct=("task_same_outcome", "mean"),
        tool_call_accuracy_consistency_pct=("tool_same_outcome", "mean"),
        step_efficiency_consistency_pct=("step_same_outcome", "mean"),
        cost_usd_stable_cases_pct=("cost_usd_within_margin", "mean"),
        elapsed_s_stable_cases_pct=("elapsed_s_within_margin", "mean"),
    ).reset_index()
    for col in table.columns:
        if col.endswith("_pct"):
            table[col] = table[col].astype(float) * 100
    table.insert(0, "group_by", dimension or "all setups")
    table.insert(1, "group", table.pop(dimension) if dimension else "all")
    return table


category_summary = pd.concat(
    [by_category(runs)] + [by_category(runs, d) for d in ["agents", "control", "policy"]],
    ignore_index=True,
)
category_summary["category"] = pd.Categorical(category_summary.category, CATEGORY_ORDER, ordered=True)
category_summary = category_summary.sort_values(["group_by", "category", "group"], key=lambda s: s if s.name != "group_by" else s.map(
    {"all setups": 0, "agents": 1, "control": 2, "policy": 3})).reset_index(drop=True)
category_summary["category"] = category_summary.category.astype(str)
category_summary.round(2)

,group_by,group,category,records,task_success_pct,tool_call_accuracy_pct,step_efficiency_pct,median_cost_usd,median_latency_s,task_success_robust_pct,tool_call_accuracy_robust_pct,step_efficiency_robust_pct,task_success_consistency_pct,tool_call_accuracy_consistency_pct,step_efficiency_consistency_pct,cost_usd_stable_cases_pct,elapsed_s_stable_cases_pct
0,all setups,all,simple lookup,192,94.27,86.98,99.48,0.0,31.07,90.10,81.25,95.83,100.0,100.0,100.0,100.0,100.00
1,all setups,all,multi-step,96,66.67,71.88,89.58,0.0,56.81,59.38,58.33,86.46,100.0,100.0,100.0,100.0,94.79
2,all setups,all,safety-and-guardrail,24,100.00,100.00,91.67,0.0,1.92,95.83,100.00,79.17,100.0,100.0,100.0,100.0,100.00
3,agents,multi,simple lookup,128,95.31,87.50,99.22,0.0,33.29,90.62,81.25,96.88,100.0,100.0,100.0,100.0,100.00
4,agents,single,simple lookup,64,92.19,85.94,100.00,0.0,25.35,89.06,81.25,93.75,100.0,100.0,100.0,100.0,100.00
5,agents,multi,multi-step,64,62.50,67.19,90.62,0.0,60.77,59.38,50.00,87.50,100.0,100.0,100.0,100.0,95.31
6,agents,single,multi-step,32,75.00,81.25,87.50,0.0,48.97,59.38,75.00,84.38,100.0,100.0,100.0,100.0,93.75
7,agents,multi,safety-and-guardrail,16,100.00,100.00,100.00,0.0,1.92,100.00,100.00,87.50,100.0,100.0,100.0,100.0,100.00
8,agents,single,safety-and-guardrail,8,100.00,100.00,75.00,0.0,1.89,87.50,100.00,62.50,100.0,100.0,100.0,100.0,100.00
9,control,agentic,simple lookup,96,94.79,87.50,98.96,0.0,33.65,93.75,80.21,93.75,100.0,100.0,100.0,100.0,100.00


## 6. Save the results

| Output | Contents |
|---|---|
| `final_evaluation_summary.csv` | One row per setup with all seven metrics |
| `final_evaluation_columns.csv` | What each summary column means and which file it comes from |
| `final_evaluation_by_category.csv` | All metrics per case category, pooled across all setups and per architecture dimension |

In [8]:
summary.to_csv(DIRECTORY / "final_evaluation_summary.csv", index=False)
guide.to_csv(DIRECTORY / "final_evaluation_columns.csv", index=False)
category_summary.to_csv(DIRECTORY / "final_evaluation_by_category.csv", index=False)
print("Saved the summary, column guide and category breakdown.")

Saved the summary, column guide and category breakdown.
